# Stop Re-running: Efficient Numerical Integration via Solver Log and Resumption

Original QMCPy demo: [`QMCPy/demos/demo_resume_data/Iteration_Log_Tolerance_Demo.ipynb`](../../../QMCPy/demos/demo_resume_data/Iteration_Log_Tolerance_Demo.ipynb)

This Julia version compares the same two workflows: solving each tolerance from scratch versus progressively resuming from the previous tolerance.


In [1]:
using QMC
using Printf


In [2]:
tols = [1e-3 / 2.0^k for k in 0:5]

function make_solver(abs_tol; seed=7)
    dd = Lattice(3; seed=seed)
    f = Genz(Uniform(dd); kind=:oscillatory, a=ones(3), u=0.5 .* ones(3))
    return CubQMCLatticeG(f; abs_tol=abs_tol, trace_iterations=true)
end

function run_fresh_curve()
    rows = NamedTuple[]
    for eps in tols
        r = integrate(make_solver(eps))
        push!(rows, (abs_tol=eps, n_total=r.data[:n_total], elapsed=r.data[:time_integrate], solution=r.solution))
    end
    return rows
end

function run_resume_curve()
    rows = NamedTuple[]
    resume_state = nothing
    cumulative = 0.0
    for eps in tols
        r = isnothing(resume_state) ? integrate(make_solver(eps)) : integrate(make_solver(eps); resume=resume_state)
        cumulative += r.data[:time_integrate]
        stop_row = iterations(r.data[:iteration_log])[end]
        push!(rows, (abs_tol=eps, n_total=r.data[:n_total], elapsed=cumulative, solution=r.solution, stop_row=stop_row))
        resume_state = r.data
    end
    return rows
end

fresh_rows = run_fresh_curve()
resume_rows = run_resume_curve()


6-element Vector{NamedTuple}:
 (abs_tol = 0.001, n_total = 16384, elapsed = 0.00021982192993164062, solution = -0.062223849929872836, stop_row = (iter = 1, n = 16384, solution = -0.062223849929872836, error_bound = 0.0003668706465500193, tol = 0.001, elapsed = 0.00021886825561523438))
 (abs_tol = 0.0005, n_total = 32768, elapsed = 0.004093647003173828, solution = -0.06236903057758148, stop_row = (iter = 1, n = 32768, solution = -0.06236903057758148, error_bound = 0.0001911878328093158, tol = 0.0005, elapsed = 0.000408172607421875))
 (abs_tol = 0.00025, n_total = 65536, elapsed = 0.008856534957885742, solution = -0.06240973714001785, stop_row = (iter = 1, n = 65536, solution = -0.06240973714001785, error_bound = 8.784764317289537e-5, tol = 0.00025, elapsed = 0.0008871555328369141))
 (abs_tol = 0.000125, n_total = 131072, elapsed = 0.01874542236328125, solution = -0.06238140666549654, stop_row = (iter = 1, n = 131072, solution = -0.06238140666549654, error_bound = 6.0376488529861935e-5, 

## Approach 2. Iteration Log with Resume Approach

The solver is executed by first running it at the loosest tolerance and then resuming at progressively tighter tolerances. Because each resumption starts from the previous state, the cumulative time and sample count grow only as needed.


In [3]:
println("Fresh solves:")
foreach(println, fresh_rows)
println("
Resume solves:")
foreach(println, resume_rows)

fresh_total_samples = sum(row.n_total for row in fresh_rows)
resume_total_samples = resume_rows[end].n_total
@printf("
Total samples when re-running from scratch: %d
", fresh_total_samples)
@printf("Samples after progressive resumption: %d
", resume_total_samples)
@printf("Sample reuse factor: %.2f×
", fresh_total_samples / resume_total_samples)


Fresh solves:
(abs_tol = 0.001, n_total = 16384, elapsed = 0.002833843231201172, solution = -0.062223849929872836)
(abs_tol = 0.0005, n_total = 16384, elapsed = 0.00032591819763183594, solution = -0.062223849929872836)
(abs_tol = 0.00025, n_total = 32768, elapsed = 0.0009489059448242188, solution = -0.06222641556727295)
(abs_tol = 0.000125, n_total = 65536, elapsed = 0.002254009246826172, solution = -0.062378451918581844)
(abs_tol = 6.25e-5, n_total = 262144, elapsed = 0.0762028694152832, solution = -0.06239521586005563)
(abs_tol = 3.125e-5, n_total = 524288, elapsed = 0.07298612594604492, solution = -0.06235929578452748)

Resume solves:
(abs_tol = 0.001, n_total = 16384, elapsed = 0.00021982192993164062, solution = -0.062223849929872836, stop_row = (iter = 1, n = 16384, solution = -0.062223849929872836, error_bound = 0.0003668706465500193, tol = 0.001, elapsed = 0.00021886825561523438))
(abs_tol = 0.0005, n_total = 32768, elapsed = 0.004093647003173828, solution = -0.06236903057758148

Below, we inspect the stored stop rows from the iteration log. These are the Julia counterpart to the reviewable iteration log used in the QMCPy notebook.


In [4]:
for row in resume_rows
    println(row.stop_row)
end

@assert resume_total_samples < fresh_total_samples
@assert all(abs(f.solution - r.solution) ≤ f.abs_tol for (f, r) in zip(fresh_rows, resume_rows))


(iter = 1, n = 16384, solution = -0.062223849929872836, error_bound = 0.0003668706465500193, tol = 0.001, elapsed = 0.00021886825561523438)
(iter = 1, n = 32768, solution = -0.06236903057758148, error_bound = 0.0001911878328093158, tol = 0.0005, elapsed = 0.000408172607421875)
(iter = 1, n = 65536, solution = -0.06240973714001785, error_bound = 8.784764317289537e-5, tol = 0.00025, elapsed = 0.0008871555328369141)
(iter = 1, n = 131072, solution = -0.06238140666549654, error_bound = 6.0376488529861935e-5, tol = 0.000125, elapsed = 0.005122184753417969)
(iter = 1, n = 262144, solution = -0.062348428790403546, error_bound = 2.4466699483028488e-5, tol = 6.25e-5, elapsed = 0.005491018295288086)
(iter = 1, n = 524288, solution = -0.06235885773195256, error_bound = 1.1271645489812593e-5, tol = 3.125e-5, elapsed = 0.012334108352661133)
